# Exp2 SFRUNet 结果分析

读取已完成正式训练的预测文件，无需 E 盘、原 cache 或重新推理。保留 Model、Baseline、True；out 是留出星系，test 是域内稀疏样本。
积分误差定义为 `100 × (预测积分−真值积分)/真值积分`，负数表示低估。默认不配置 burst 窗口。

In [ ]:
# 配置：修改结果目录；路径可绝对指定或相对于项目根目录。
from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Experiment" / "SFRUNet.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("请从项目目录或其子目录打开 notebook。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULT_DIR = PROJECT_ROOT / "Results" / "请替换为已完成的SFRUNet结果目录"
MAKE_PLOTS = True
GAP_FACTOR = 5.0
BURST_WINDOW = None       # 例如 (起始时间, 结束时间)，单位 Gyr，必须自行核对
TIME_OFFSET = 0.0         # 分析时间 = snapshot 时间 + 此偏移
BURST_THRESHOLD = None    # 可选事件阈值，单位 Msun/yr；不自动套 Model0.75 参数


In [ ]:
# 执行分析：每次建立独立输出目录，不覆盖既有图表，不加载模型。
import importlib
import pandas as pd
from IPython.display import display, Image
from VisualScripts.ResultAnalysis import analyze_sfr_unet

importlib.reload(analyze_sfr_unet)
result = analyze_sfr_unet.analyze_result(
    RESULT_DIR, plots=MAKE_PLOTS, gap_factor=GAP_FACTOR,
    burst_window=BURST_WINDOW, time_offset=TIME_OFFSET,
    burst_threshold=BURST_THRESHOLD,
)
ANALYSIS_DIR = Path(result["output_dir"])
print("分析输出：", ANALYSIS_DIR)


In [ ]:
# 总量与逐格指标：test/out 分开，物理 MAE/RMSE 单位均为 Msun/yr；逐格量已积分。
metrics = result["metrics"]
display(metrics)
held_out_metrics = metrics[metrics["split"] == "out"].copy()
display(held_out_metrics)

# 运行等权摘要与按样本汇总不同；R2 不做无解释的跨运行平均。
display(metrics.groupby(["split", "method"])[["MAE", "RMSE", "Cell_MAE", "Cell_RMSE", "NegativePredFrac"]].mean())


In [ ]:
# 留出轨迹累计误差：正数高估、负数低估；积分范围为缓存覆盖区间，不包含初始已有恒星。
integrated = result["integrated_metrics"]
display(integrated)
print("Model Error all time / Baseline Error all time 的单位为百分比。")
print("请同时查看起止时间、covered_duration_gyr 和 excluded_gap_count。")


In [ ]:
# 显示瞬时、累计 SFR，以及局部误差抵消图；test 不被当作完整历史积分。
if MAKE_PLOTS:
    for pattern in ("out_*/sfr_history.png", "out_*/cumulative_sfr.png", "out_*/scatter_and_cell_error.png", "learning_curves.png"):
        for path in sorted(ANALYSIS_DIR.glob(pattern)):
            display(Image(filename=str(path)))


In [ ]:
# 空间对比：默认展示留出轨迹首帧、真实总 SFR 峰值帧和末帧。
# 横纵为 r/theta 单元索引，不是等面积的物理坐标。
if MAKE_PLOTS:
    for path in sorted(ANALYSIS_DIR.glob("out_*/maps_*.png")):
        display(Image(filename=str(path)))


In [ ]:
# 可选 burst 分析：启用需在配置 cell 指定窗口；事件时刻还需显式阈值。
if BURST_WINDOW is None:
    print("尚未配置 burst 窗口；未应用任何论文时间或自动阈值。")
else:
    display(result["burst_metrics"])
    print("mass_relative_error / peak_relative_error 是比例，不是百分数。")
    print("开始/结束误差只在两者事件均检出且对应边界未删失时报告。")


## 异常帧及标准化输入检查（可选）
检查cache输入及训练分布；不读取原始HDF，不重新训练或推理。全局/同位置范围只描述训练覆盖，不自动判定物理错误。

In [ ]:
# 异常帧检查配置：默认关闭，需本地具有与结果对应的 train/out 标量cache。
INSPECT_INPUTS = False
INPUT_CACHE_DIR = PROJECT_ROOT / ".cache"
INPUT_TIMES = [8.99375, 9.405, 12.44125]
INPUT_NEIGHBORS = 5
INPUT_CELLS = None  # 或 [(6, 0), (9, 2), (9, 1)]，与三个时间对应
INPUT_CHANNELS = None  # None检查全部通道，也可指定通道名列表


In [ ]:
# 按UID读取输入，输出邻帧、训练分位和局部最近训练样本；不覆盖旧检查。
if INSPECT_INPUTS:
    from VisualScripts.ResultAnalysis import inspect_sfr_unet_inputs
    importlib.reload(inspect_sfr_unet_inputs)
    input_review = inspect_sfr_unet_inputs.inspect_inputs(
        RESULT_DIR, INPUT_CACHE_DIR, times=INPUT_TIMES, neighbors=INPUT_NEIGHBORS,
        cells=INPUT_CELLS, channels=INPUT_CHANNELS, plots=True)
    INPUT_REVIEW_DIR = Path(input_review["output_dir"])
    display(input_review["frames"])
    display(input_review["points"])
    display(input_review["nearest"])
    print("输入检查目录：", INPUT_REVIEW_DIR)
else:
    print("输入检查未启用。")


In [ ]:
# 展示邻帧气体输入地图与选定格子的时序；阴影是训练集同位置p01-p99。
if INSPECT_INPUTS:
    for path in sorted(INPUT_REVIEW_DIR.glob("event*/*.png")):
        display(Image(filename=str(path)))
